# Notebook 0 — Data Wrangling final

Este notebook reconstruye el archivo **`csv/data_wramling_final.csv`** que consume
`notebook1.ipynb`. Toma el dataset crudo del scraping
(`data/data_wrangling_20250324_202229.csv`, 3 739 avisos de alquiler de
departamentos en Lima) y lo deja **apto para el análisis**.

**Transformaciones aplicadas**
1. Se descartan columnas de texto libre no usadas en el análisis (`descripcion`, `direccion`).
2. Se eliminan filas con nulos en las variables núcleo (`dormitorios`, `banos`, `area`, `precio`, `distrito`), necesarias para el clustering y el precio por m².
3. Se derivan las columnas que el análisis espera:
   - **`precioxm2`** = `precio / area`.
   - **`fecha_publicacion`** binaria: `0` = publicado este mes (≤ 30 días), `1` = mes(es) anteriores.
   - **`antiguedad_categoria`**: `Grupo 1`–`Grupo 4` por cuartiles de antigüedad (Grupo 1 = más reciente) y `desconocido` para antigüedad faltante.
4. Se guarda con índice, de modo que `notebook1` pueda hacer `drop(columns=['Unnamed: 0'])`.


In [1]:
import pandas as pd
import numpy as np

## 1. Cargar el dataset crudo

In [2]:
raw = pd.read_csv("../data/data_wrangling_20250324_202229.csv")
print("crudo:", raw.shape)
raw.head()

crudo: (3739, 13)


,longitud,descripcion,area,banos,precio,latitud,dormitorios,fecha_publicacion,distrito,antiguedad,direccion,precision,type
0,-77.018606,Se Alquila Lindo Departamento en Barranco\n¡De...,77.0,2.0,2960.0,-12.139912,3.0,0.0,barranco,5.0,"Ca. Enrique Barrón 280, Barranco 15063, Peru",ROOFTOP,street_address
1,-77.021154,Se Alquila Departamento Duplex en Corazón de B...,96.0,2.0,2250.0,-12.139272,2.0,1.0,barranco,0.0,"Jr. Tumbes 306, Lima 15063, Peru",ROOFTOP,street_address
2,-77.023879,Se Alquila Lindo Departamento en Barranco\nSE ...,40.0,1.0,2000.0,-12.139555,1.0,1.0,barranco,8.0,"Av. el Sol Oeste 203 - 504, Barranco 15063, Peru",ROOFTOP,establishment
3,-77.017627,Departamento de 3 Habitaciones Sin Amoblar en ...,80.0,2.0,2974.0,-12.138541,3.0,1.0,barranco,10.0,"Av. República de Panamá 330, Barranco 15049, Peru",ROOFTOP,establishment
4,-77.025239,Se Alquila Lindo Departamento en La Av El Sol ...,70.0,1.0,2800.0,-12.138338,2.0,1.0,barranco,5.0,"Edificio El Sol 450, Av. el Sol Oeste 448, Bar...",ROOFTOP,street_address


## 2. Limpieza: columnas de texto y nulos núcleo

In [3]:
df = raw.drop(columns=["descripcion", "direccion"])
df = df.dropna(subset=["dormitorios", "banos", "area", "precio", "distrito"]).copy()
df["dormitorios"] = df["dormitorios"].astype(int)
df["banos"] = df["banos"].astype(int)
print("tras limpieza:", df.shape)

tras limpieza: (3516, 11)


## 3. Variables derivadas

In [4]:
# 3.1 Precio por metro cuadrado
df["precioxm2"] = df["precio"] / df["area"]

# 3.2 fecha_publicacion -> binaria (0 = este mes, 1 = meses anteriores)
df["fecha_publicacion"] = np.where(df["fecha_publicacion"].fillna(0) <= 30, 0, 1)

# 3.3 antiguedad_categoria por cuartiles (Grupo 1 = mas reciente); NaN -> 'desconocido'
q1, q2, q3 = df["antiguedad"].quantile([0.25, 0.50, 0.75])
bins = [-np.inf, q1, q2, q3, np.inf]
etiquetas = ["Grupo 1", "Grupo 2", "Grupo 3", "Grupo 4"]
df["antiguedad_categoria"] = pd.cut(df["antiguedad"], bins=bins, labels=etiquetas).astype(object)
df["antiguedad_categoria"] = df["antiguedad_categoria"].fillna("desconocido")

print("cuartiles antiguedad:", q1, q2, q3)
df["antiguedad_categoria"].value_counts(dropna=False)

cuartiles antiguedad: 3.0 8.0 14.0


antiguedad_categoria
desconocido    1036
Grupo 1         751
Grupo 4         601
Grupo 3         568
Grupo 2         560
Name: count, dtype: int64

## 4. Verificación del esquema esperado por `notebook1`

In [5]:
num = df.select_dtypes(include=["number"]).columns.tolist()
obj = df.select_dtypes(exclude=["number"]).columns.tolist()
print("numericas (%d):" % len(num), num)
print("categoricas (%d):" % len(obj), obj)
assert len(num) == 9 and set(obj) == {"distrito", "precision", "type", "antiguedad_categoria"}
print("Esquema OK -> grilla 3x3 numericas + 4 categoricas")

numericas (9): ['longitud', 'area', 'banos', 'precio', 'latitud', 'dormitorios', 'fecha_publicacion', 'antiguedad', 'precioxm2']
categoricas (4): ['distrito', 'precision', 'type', 'antiguedad_categoria']
Esquema OK -> grilla 3x3 numericas + 4 categoricas


## 5. Guardar el CSV final

Se guarda con índice para que `notebook1` haga `drop(columns=['Unnamed: 0'])`.

In [6]:
df = df.reset_index(drop=True)
df.to_csv("../csv/data_wramling_final.csv", index=True)
print("Guardado csv/data_wramling_final.csv ->", df.shape)

Guardado csv/data_wramling_final.csv -> (3516, 13)
